# Calculus, Gradients, and Optimization
Ava studied two hours and completed three practice sets. Our model predicted 55. Suppose she scored 60 on a fictional exam. Which numbers inside the model should change, and by how much? Run the cells in order in a CPU Colab session.

In [ ]:
import torch

x = torch.tensor([2., 3.], dtype=torch.float64)
w = torch.tensor([10., 5.], dtype=x.dtype, requires_grad=True)
b = torch.tensor(20., dtype=x.dtype, requires_grad=True)
y = x.new_tensor(60.)
sigma = x.new_tensor(10.)
mu = x @ w + b
print("Predicted mean:", mu.item(), "Observed:", y.item())

## Score the original prediction
Module 02 assumed a Gaussian with standard deviation 10 points. Keep the input, observation, and spread fixed. The single-observation negative log-likelihood (NLL) is L = log(σ√(2π)) + (μ − y)²/(2σ²). The first term stays constant as μ changes. The second adds 25/200 = 0.125 at μ = 55.

In [ ]:
def nll(mean):
    return -torch.distributions.Normal(mean, sigma).log_prob(y)

loss = nll(mu)
print(f"Initial NLL: {loss.item():.6f}")

## Find the slope by hand
Differentiate the squared residual with respect to μ. The exponent supplies a factor of 2, and the derivative of μ − y is 1. The result is (μ − y)/σ² = −5/100 = −0.05. A small increase in μ decreases the loss. Now trace that slope through μ = 2w₁ + 3w₂ + b. Holding the other parameters fixed gives local slopes 2, 3, and 1. Multiplication gives the three partial derivatives.

In [ ]:
slope = (mu.detach() - y) / sigma.square()
dw = slope * x
db = slope
print("Mean, weight and bias slopes:", slope.item(), dw.tolist(), db.item())

## Check the same computation with autograd
PyTorch recorded the operations on w and b during the forward pass. Calling backward() on the scalar loss applies the chain rule along that graph. Each assertion compares a computed derivative to the result from the preceding cell.

In [ ]:
mu.retain_grad()  # Keep the intermediate slope so we can inspect it.
loss.backward()
torch.testing.assert_close(mu.grad, slope)
torch.testing.assert_close(w.grad, dw)
torch.testing.assert_close(b.grad, db)
print("Autograd:", mu.grad.item(), w.grad.tolist(), b.grad.item())

## Take one simultaneous descent step
Subtract each gradient times a learning rate of 1.0. The two weights should become 10.10 and 5.15, and the bias 20.05. Predict the new mean before running the cell. Use no_grad() to perform the update outside the differentiation graph. To repeat training, rebuild the forward pass and clear accumulated gradients at the start of each step.

In [ ]:
eta = 1.0
with torch.no_grad():
    w -= eta * w.grad
    b -= eta * b.grad
    updated_mean = x @ w + b
    updated_loss = nll(updated_mean)

print("New weights and bias:", w.tolist(), b.item())
print(f"Mean: {mu.item():.2f} -> {updated_mean.item():.2f}")
print(f"NLL: {loss.item():.6f} -> {updated_loss.item():.6f}")
torch.testing.assert_close(updated_mean, x.new_tensor(55.7))
assert updated_loss < loss

## Explain the size of the correction
Ava's mean gains 2(0.10) + 3(0.15) + 0.05 = 0.70 points. If e = μ − y, then the next error equals e times 1 − η(2² + 3² + 1)/100. A multiplier between 0 and 1 shrinks the error from the same side. Between −1 and 0 it shrinks while crossing the target. At −1 the error alternates without shrinking. Beyond that boundary the error grows.

In [ ]:
import torch

x = torch.tensor([2., 3.], dtype=torch.float64)
y = x.new_tensor(60.)
for eta in [0., 1., 100/14, 10., 100/7, 16.]:
    trial_w = x.new_tensor([10., 5.], requires_grad=True)
    trial_b = x.new_tensor(20., requires_grad=True)
    errors = []
    for _ in range(3):
        mean = x @ trial_w + trial_b
        trial_loss = -torch.distributions.Normal(mean, 10.).log_prob(y)
        dw, db = torch.autograd.grad(trial_loss, (trial_w, trial_b))
        with torch.no_grad():
            trial_w -= eta * dw
            trial_b -= eta * db
            errors.append(round((x @ trial_w + trial_b - y).item(), 3))
    print(f"Rate {eta:.3f}: errors after three steps {errors}")

## Practice with a new observation
Reset the weights to [10, 5] and the bias to 20. Suppose the observed score is 50 instead of 60. Predict the signs of all three gradients and the mean after one step at rate 1. Then check below. Explain why success on this single training observation leaves performance on other students open.

In [ ]:
test_w = x.new_tensor([10., 5.], requires_grad=True)
test_b = x.new_tensor(20., requires_grad=True)
test_mean = x @ test_w + test_b
test_loss = -torch.distributions.Normal(test_mean, sigma).log_prob(x.new_tensor(50.))
test_loss.backward()
with torch.no_grad():
    next_mean = x @ (test_w - test_w.grad) + (test_b - test_b.grad)
torch.testing.assert_close(next_mean, x.new_tensor(54.3))
print(test_w.grad.tolist(), test_b.grad.item(), next_mean.item())